# **Modelos NLP**

In [3]:
# [Config]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader


import time, os, ast
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

Usando dispositivo: cpu


In [4]:
# [Data]
df = pd.read_csv('https://raw.githubusercontent.com/Darally06/NLP-Jarvis-Hiring/refs/heads/main/Data/Clean_words.csv')
df = df.rename(columns={'macro_label': 'Category'})
print(f"Registros totales: {len(df)}")
df.head()


Registros totales: 2483


,Category,Resume_str,clean_text,tokens,bert_text,fasttext_text
0,Servicios Profesionales y Públicos,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,administrator marketing associate administrato...,"['administrator', 'marketing', 'associate', 'a...",hr administratormarketing associate\n...,__label__HR administrator marketing associate ...
1,Servicios Profesionales y Públicos,"HR SPECIALIST, US HR OPERATIONS ...",specialist operation summary versatile medium ...,"['specialist', 'operation', 'summary', 'versat...",hr specialist us hr operations ...,__label__HR specialist operation summary versa...
2,Servicios Profesionales y Públicos,HR DIRECTOR Summary Over 2...,director summary year experience recruiting pl...,"['director', 'summary', 'year', 'experience', ...",hr director summary over ...,__label__HR director summary year experience r...
3,Servicios Profesionales y Públicos,HR SPECIALIST Summary Dedica...,specialist summary dedicated driven dynamic ye...,"['specialist', 'summary', 'dedicated', 'driven...",hr specialist summary dedica...,__label__HR specialist summary dedicated drive...
4,Servicios Profesionales y Públicos,HR MANAGER Skill Highlights ...,manager skill highlight skill department start...,"['manager', 'skill', 'highlight', 'skill', 'de...",hr manager skill highlights ...,__label__HR manager skill highlight skill depa...


In [5]:
def ensure_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)  # convierte el texto "['a','b']" → ['a','b']
        except:
            return x.split()  # si no tiene formato de lista, separa por espacios
    elif isinstance(x, list):
        return x
    else:
        return []
df['tokens'] = df['tokens'].apply(ensure_list)


In [6]:
# 3. División estratificada
train_val, test = train_test_split(
    df, test_size=0.15, stratify=df['Category'], random_state=SEED
)
train, val = train_test_split(
    train_val, test_size=0.17647,  # 0.17647 * 0.85 ≈ 0.15 total
    stratify=train_val['Category'],
    random_state=SEED
)
print("División de grupos de datos")
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

# Codificar etiquetas
train['label'] = train['Category'].astype('category').cat.codes
val['label'] = val['Category'].astype('category').cat.codes
test['label'] = test['Category'].astype('category').cat.codes

num_labels = train['label'].nunique()
print("Número de clases:", num_labels)


División de grupos de datos
Train: 1737 | Val: 373 | Test: 373
Número de clases: 5


In [7]:
# [Def]

# Pesos de clase balanceados
def get_class_weights(train_df, device):
    """Calcula pesos de clase balanceados"""
    # Calcula los pesos inversamente proporcionales a la frecuencia de cada clase
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_df['label']),
        y=train_df['label']
    )
    # Convierte los pesos a tensores de PyTorch
    return torch.tensor(class_weights, dtype=torch.float).to(device)

# Evaluar modelos
def evaluate_and_save(model, data_loader, device, base_dir, combo_name):
    """Evalúa el modelo y guarda métricas dentro de su carpeta específica."""
    model.eval()
    combo_dir = os.path.join(base_dir, combo_name)
    os.makedirs(combo_dir, exist_ok=True)

    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc=f"Evaluando {combo_name}"):
            if isinstance(batch, dict):
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                logits = getattr(outputs, "logits", outputs)
                preds = logits.argmax(dim=1)
                labels = batch.get("labels").cpu().numpy()
            elif isinstance(batch, (list, tuple)) and len(batch) == 2:
                X_batch, y_batch = batch
                X_batch, y_batch = X_batch.to(device), y_batch.to(device).long()
                outputs = model(X_batch)
                preds = outputs.argmax(dim=1)
                labels = y_batch.cpu().numpy()
            else:
                raise TypeError(f"Tipo de batch no reconocido: {type(batch)}")

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels)

    # ---- Métricas ----
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
    df_report = pd.DataFrame(report).transpose()
    conf_mat = confusion_matrix(all_labels, all_preds)

    # ---- Guardar ----
    df_report.to_csv(f"{combo_dir}/report.csv", index=True)
    pd.DataFrame(conf_mat).to_csv(f"{combo_dir}/confusion_matrix.csv", index=False)

    with open(f"{combo_dir}/metrics.txt", "w") as f:
        f.write(f"Accuracy: {acc:.4f}\n")
        f.write(df_report.to_string())
        f.write("\n\nConfusion Matrix:\n")
        f.write(pd.DataFrame(conf_mat).to_string())

    return acc



## DistilBERT

In [ ]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments, Trainer
)
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Etiquetado y selección de texto para BERT
train['text'] = train['bert_text']
val['text'] = val['bert_text']
test['text'] = test['bert_text']

In [ ]:
# Cargar el tokenizer [sin distinguir entre Mayús-Minus] con long max de 512
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
MAX_LEN = 512

class BertDataset(Dataset):
    # Tokenizar textos, truncar, padding hasta max_length
    def __init__(self, df, tokenizer):
        self.encodings = tokenizer(
            df['text'].tolist(),
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN
        )
        self.labels = df['label'].tolist()
    # Diccionario de tensores
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BertDataset(train, tokenizer)
eval_dataset = BertDataset(val, tokenizer)

In [ ]:
# Modelo y pesos
# Modelo para clasificacion de secuencias, con n clases.
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

class_weights = get_class_weights(train, device)    # Calcular pesos para balanceo
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights) # Función de perdida

from transformers import Trainer

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = loss_fn(outputs.logits, labels)  # loss_fn usa tus class_weights
        return (loss, outputs) if return_outputs else loss

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    logging_strategy="steps",
    logging_dir="./logs",
    use_cpu=True,
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

## BiLSTM [Word2Vec]

In [8]:
!pip install gensim

In [9]:
from gensim.models import Word2Vec
from tqdm import tqdm

# Etiquetado y selección de texto para Word2Vec
train['text'] = train['tokens']
val['text'] = val['tokens']
test['text'] = test['tokens']

train_df = train.reset_index(drop=True)
val_df = val.reset_index(drop=True)
test_df = test.reset_index(drop=True)

# Preparar lista de oraciones (listas de tokens) para Word2Vec
all_sentences = train_df['tokens'].tolist() + val_df['tokens'].tolist() + test_df['tokens'].tolist()

In [25]:
class Word2VecDataset(Dataset):
    def __init__(self, df, w2v_model, sequence_length):
        self.data = df['tokens'].tolist()
        self.labels = df['label'].tolist()
        self.w2v_model = w2v_model
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = self.data[idx]
        # Convert tokens to their Word2Vec embeddings
        embeddings = [self.w2v_model.wv[token] for token in tokens if token in self.w2v_model.wv]

        # Pad or truncate sequences
        if len(embeddings) < self.sequence_length:
            # Pad with zeros
            padding = [np.zeros(self.w2v_model.vector_size)] * (self.sequence_length - len(embeddings))
            padded_embeddings = embeddings + padding
        else:
            # Truncate
            padded_embeddings = embeddings[:self.sequence_length]

        # Convert to torch tensor
        input_tensor = torch.tensor(padded_embeddings, dtype=torch.float)
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.long)

        return input_tensor, label_tensor

class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, num_layers, dropout, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0 )
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.dropout(lstm_out[:, -1, :]) # último hidden state
        out = self.fc(out)
        return out

In [21]:
def train_bilstm(model, train_loader, val_loader, criterion, optimizer, device,
                 epochs, patience=3):
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'epoch_time': []}
    best_val_loss = float("inf")
    patience_counter = 0

    print(f"Entrenando modelo ({epochs} épocas) ...")
    total_start = time.time()

    for epoch in range(epochs):
        torch.cuda.empty_cache()
        epoch_start = time.time()

        model.train()
        total_loss = 0

        for X_batch, y_batch in tqdm(
            train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False
            ):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device).long()
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        # --- Validación ---
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val, y_val = X_val.to(device), y_val.to(device).long()
                outputs = model(X_val)
                loss = criterion(outputs, y_val)
                val_loss += loss.item()
                preds = outputs.argmax(dim=1)
                correct += (preds == y_val).sum().item()
                total += y_val.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct / total

        # --- Tiempos ---
        epoch_time = time.time() - epoch_start
        history['epoch_time'].append(epoch_time)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1:>2}/{epochs} | "
              f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
              f"Acc: {val_acc:.4f} | Tiempo: {epoch_time:.2f} s")

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping activado.")
                break

    total_time = time.time() - total_start
    print(f"Entrenamiento completado en {total_time/60:.2f} minutos totales.")
    history['total_time_min'] = total_time / 60
    return history


In [26]:
def run_experiment(train, val, test, params, model_name, w2v):
    """
    Ejecuta un experimento BiLSTM con una configuración de hiperparámetros dada.
    Reutiliza un modelo Word2Vec ya entrenado o cargado externamente.
    """

    # --- Directorios ---
    base_dir = f"results/{model_name}"
    combo_name = f"BiLSTM_{params_to_name(params)}"
    combo_dir = os.path.join(base_dir, combo_name)
    os.makedirs(combo_dir, exist_ok=True)

    # --- Configuración fija ---
    embedding_dim = w2v.vector_size
    sequence_length = params['sequence_length'] # Use sequence_length from params
    batch_size = params['batch_size'] # Use batch_size from params
    dropout_rate = params['dropout_rate'] # Use dropout_rate from params

    # --- Datasets y DataLoaders ---
    train_ds = Word2VecDataset(train, w2v, sequence_length)
    val_ds = Word2VecDataset(val, w2v, sequence_length)
    test_ds = Word2VecDataset(test, w2v, sequence_length)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    # --- Modelo ---
    num_classes = train['label'].nunique()

    model = BiLSTMClassifier(
        embedding_dim=params['embedding_dim'],
        hidden_dim=params['lstm_units'],
        num_layers=params['num_lstm_layers'],
        dropout=params['dropout_rate'],
        num_classes=num_classes
        ).to(device)

    # --- Configuración de entrenamiento ---
    class_weights = get_class_weights(train, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])

    # --- Entrenamiento ---
    history = train_bilstm(
        model, train_loader, val_loader, criterion, optimizer, device, epochs
    )

    # --- Evaluación y guardado ---
    # Assuming evaluate_and_save is already defined
    acc = evaluate_and_save(
        model, test_loader, device, base_dir, combo_name # Pass device here
    )
    print(f"Accuracy final: {acc:.4f}")

    # --- Guardar modelo ---
    model_path = os.path.join(combo_dir, "model.pt")
    torch.save(model.state_dict(), model_path)

    return acc, history

In [30]:
# -----
# INICIO
# -----

#
from itertools import product # Import product here

def params_to_name(params):
    """Crea nombre legible y único a partir de los parámetros."""
    return (f"lstm{params['lstm_units']}_layers{params['num_lstm_layers']}"
            f"_lr{params['learning_rate']}_drop{params['dropout_rate']}"
            f"_batch{params['batch_size']}_seq{params['sequence_length']}")

results = []
model_name = "BiLSTM"
base_dir = f"results/{model_name}"
os.makedirs(base_dir, exist_ok=True)

# Hiperparámtros
param_grid = {
    "lstm_units": [32, 64],
    "num_lstm_layers": [2, 3],
    "learning_rate": [0.004, 0.005],
    "batch_size": [8],
    "sequence_length": [50],
    "dropout_rate": [0.3],
    "embedding_dim": [200]
}
# Fijos
epochs = 50


# Generar combinaciones de hiperparámetros
param_combinations = [dict(zip(param_grid.keys(), v)) for v in product(*param_grid.values())]

# Modelo Word2Vec
w2v_path = os.path.join(base_dir, f"w2v_{embedding_dim}.model")

if os.path.exists(w2v_path):
    print(f"Cargando modelo Word2Vec desde {w2v_path}")
    w2v = Word2Vec.load(w2v_path)
else:
    print(f"Entrenando nuevo modelo Word2Vec (dim={embedding_dim})...")
    all_sentences = train['tokens'].tolist() + val['tokens'].tolist()
    w2v = Word2Vec(
        sentences=all_sentences,
        vector_size=embedding_dim,
        window=5,
        min_count=2,
        workers=1,
        epochs=10,
        seed=SEED
    )
    w2v.save(w2v_path)
    print(f" Word2Vec guardado en {w2v_path}")

Cargando modelo Word2Vec desde results/BiLSTM/w2v_200.model


In [ ]:
from sklearn.model_selection import ParameterGrid
# Ejecutar experimentos ===
for params in param_combinations:
    acc, history = run_experiment(train, val, test, params, "BiLSTM_grid", w2v)
    results.append({**params, "accuracy": acc})

# === Guardar resumen global ===
summary_df = pd.DataFrame(results)
summary_path = "results/BiLSTM_grid/summary_experiments.csv"
summary_df.to_csv(summary_path, index=False)

print("\n✅ Todos los experimentos completados.")
print("Top 5 combinaciones por accuracy:")
print(summary_df.sort_values(by="accuracy", ascending=False).head(5).round(4))

Entrenando modelo (50 épocas) ...


Epoch 1/50:   0%|          | 0/218 [00:00<?, ?it/s]/tmp/ipython-input-3068726197.py:26: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  input_tensor = torch.tensor(padded_embeddings, dtype=torch.float)


Epoch  1/50 | Train: 1.5116 | Val: 1.3822 | Acc: 0.4370 | Tiempo: 12.10 s


Epoch  2/50 | Train: 1.2713 | Val: 1.2316 | Acc: 0.5737 | Tiempo: 11.71 s


Epoch  3/50 | Train: 1.1775 | Val: 1.1593 | Acc: 0.5710 | Tiempo: 11.69 s


Epoch  4/50 | Train: 1.0495 | Val: 1.1527 | Acc: 0.5657 | Tiempo: 13.37 s


Epoch  5/50 | Train: 1.0078 | Val: 1.1123 | Acc: 0.5818 | Tiempo: 16.07 s


Epoch 6/50:  55%|█████▌    | 120/218 [00:05<00:05, 19.00it/s]

## CNN-1D

In [ ]:
# Etiquetado y selección de texto para
train['text'] = train['']
val['text'] = val['']
test['text'] = test['']

## TF-IDF [XGBoost]

In [ ]:
# Etiquetado y selección de texto para
train['text'] = train['clean_text']
val['text'] = val['clean_text']
test['text'] = test['clean_text']

## FastTest

In [ ]:
# Etiquetado y selección de texto para
train['text'] = train['fasttext_text']
val['text'] = val['fasttext_text']
test['text'] = test['fasttext_text']